Identify the most common routes

In [0]:
from pyspark.sql import functions as F

items = spark.table("workspace.default.silver_order_items")
orders = spark.table("workspace.default.silver_orders")
customers = spark.table("workspace.default.silver_customers")
sellers = spark.table("workspace.default.silver_sellers")

route_pairs = (
    items
    .join(orders.select("order_id", "customer_id"), "order_id")
    .join(
        customers.select(
            "customer_id",
            F.col("customer_zip_code_prefix").alias("customer_zip")
        ),
        "customer_id"
    )
    .join(
        sellers.select(
            "seller_id",
            F.col("seller_zip_code_prefix").alias("seller_zip")
        ),
        "seller_id"
    )
    .groupBy("seller_zip", "customer_zip")
    .agg(F.countDistinct("order_id").alias("shipment_count"))
    .orderBy(F.desc("shipment_count"))
)

print("Distinct seller-customer postal routes:", route_pairs.count())

display(route_pairs.limit(20))

Distinct seller-customer postal routes: 93939


seller_zip,customer_zip,shipment_count
81130,80030,15
14940,28970,15
14940,35162,14
14940,22793,13
14940,13212,12
14940,11065,10
14940,22775,10
14940,29101,9
14940,13290,9
14940,33200,9


Inspect the geolocation table

In [0]:
geo = spark.table("workspace.default.silver_geolocation")

geo.printSchema()

display(geo.limit(5))

root
 |-- zip_prefix: string (nullable = true)
 |-- latitude: double (nullable = true)
 |-- longitude: double (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- location_records: long (nullable = true)



zip_prefix,latitude,longitude,city,state,location_records
01017,-23.549531271947856,-46.63102384555222,são paulo,SP,41
01204,-23.53599263946852,-46.64408982260721,sao paulo,SP,38
01251,-23.548374516404742,-46.67484647170288,são paulo,SP,18
01307,-23.552778010115734,-46.65320770049066,sao paulo,SP,311
01322,-23.56317165882957,-46.643456141575236,sao paulo,SP,139


Attach coordinates to the 50 most common routes

In [0]:
geo = spark.table("workspace.default.silver_geolocation")

seller_geo = geo.select(
    F.col("zip_prefix").alias("seller_zip"),
    F.col("latitude").alias("seller_lat"),
    F.col("longitude").alias("seller_lon")
)

customer_geo = geo.select(
    F.col("zip_prefix").alias("customer_zip"),
    F.col("latitude").alias("customer_lat"),
    F.col("longitude").alias("customer_lon")
)

route_coordinates = (
    route_pairs
    .limit(50)
    .join(seller_geo, "seller_zip", "inner")
    .join(customer_geo, "customer_zip", "inner")
    .filter(
        F.col("seller_lat").isNotNull() &
        F.col("customer_lat").isNotNull()
    )
)

print("Routes with coordinates:", route_coordinates.count())

display(route_coordinates)

Routes with coordinates: 50


customer_zip,seller_zip,shipment_count,seller_lat,seller_lon,customer_lat,customer_lon
6950,14940,6,-21.757320637712766,-48.82974353343713,-23.941873053352303,-47.07320010425571
11740,14940,8,-21.757320637712766,-48.82974353343713,-24.18271029899563,-46.80183859998844
12233,13405,7,-22.708702100662567,-47.664700852471455,-23.246680687314193,-45.88690913568317
13290,14940,9,-21.757320637712766,-48.82974353343713,-23.08550507419915,-46.95954096067056
21235,14940,6,-21.757320637712766,-48.82974353343713,-22.837911776525186,-43.322271046058205
22790,14940,8,-21.757320637712766,-48.82974353343713,-23.012986504226507,-43.46551954103294
13214,14940,6,-21.757320637712766,-48.82974353343713,-23.157994972014652,-46.918199852685866
29101,14940,9,-21.757320637712766,-48.82974353343713,-20.345030195796223,-40.28790374199651
11065,14940,10,-21.757320637712766,-48.82974353343713,-23.965227243543822,-46.34358134349832
11075,14940,7,-21.757320637712766,-48.82974353343713,-23.95460047631286,-46.3377311534732


In [0]:
import requests

test_route = route_coordinates.first()

seller_lon = test_route["seller_lon"]
seller_lat = test_route["seller_lat"]
customer_lon = test_route["customer_lon"]
customer_lat = test_route["customer_lat"]

url = (
    f"https://router.project-osrm.org/route/v1/driving/"
    f"{seller_lon},{seller_lat};{customer_lon},{customer_lat}"
    f"?overview=false"
)

response = requests.get(url, timeout=30)

print("HTTP Status:", response.status_code)

route_data = response.json()

if route_data["code"] == "Ok":
    route = route_data["routes"][0]
    print("Distance (km):", round(route["distance"] / 1000, 2))
    print("Estimated driving duration (hours):", round(route["duration"] / 3600, 2))
else:
    print("Routing error:", route_data["code"])

HTTP Status: 200
Distance (km): 395.05
Estimated driving duration (hours): 5.04


Fetch all 50 routes

In [0]:
import requests
import json
import time

route_records = []

for row in route_coordinates.collect():
    seller_lon = row["seller_lon"]
    seller_lat = row["seller_lat"]
    customer_lon = row["customer_lon"]
    customer_lat = row["customer_lat"]

    url = (
        f"https://router.project-osrm.org/route/v1/driving/"
        f"{seller_lon},{seller_lat};{customer_lon},{customer_lat}"
        f"?overview=false"
    )

    try:
        response = requests.get(url, timeout=30)
        data = response.json()

        if response.status_code == 200 and data["code"] == "Ok":
            route = data["routes"][0]

            route_records.append({
                "seller_zip": row["seller_zip"],
                "customer_zip": row["customer_zip"],
                "shipment_count": row["shipment_count"],
                "distance_meters": route["distance"],
                "duration_seconds": route["duration"],
                "osrm_code": data["code"]
            })

        else:
            print("Failed:", row["seller_zip"], row["customer_zip"], data.get("code"))

    except Exception as e:
        print("Error:", row["seller_zip"], row["customer_zip"], str(e))

    time.sleep(0.2)

print("Successful routes:", len(route_records))
print("Failed routes:", 50 - len(route_records))


raw_routes_path = "/Volumes/workspace/default/logisense_raw/routes/"
dbutils.fs.mkdirs(raw_routes_path)

dbutils.fs.put(
    raw_routes_path + "osrm_routes.json",
    json.dumps(route_records, indent=2),
    True
)

print("Raw routing data saved successfully!")


Successful routes: 50
Failed routes: 0
Wrote 8673 bytes.
Raw routing data saved successfully!


In [0]:
import json

raw_routes_path = "/Volumes/workspace/default/logisense_raw/routes/"

dbutils.fs.mkdirs(raw_routes_path)

route_json = json.dumps(route_records, indent=2)

dbutils.fs.put(
    raw_routes_path + "osrm_routes.json",
    route_json,
    True
)

print("Successful routes:", len(route_records))
print("Raw routing JSON saved successfully!")

display(dbutils.fs.ls(raw_routes_path))

Wrote 8673 bytes.
Successful routes: 50
Raw routing JSON saved successfully!


path,name,size,modificationTime
dbfs:/Volumes/workspace/default/logisense_raw/routes/osrm_routes.json,osrm_routes.json,8673,1794934252000


Bronze Routing table

In [0]:
routes_bronze = (
    spark.read
    .option("multiline", True)
    .json("/Volumes/workspace/default/logisense_raw/routes/osrm_routes.json")
    .withColumn("distance_km", F.round(F.col("distance_meters") / 1000, 2))
    .withColumn("estimated_duration_hours", F.round(F.col("duration_seconds") / 3600, 2))
)

routes_bronze.write.format("delta").mode("overwrite").saveAsTable(
    "workspace.default.bronze_routes"
)

print("Bronze Routing table created!")
print("Total routes:", routes_bronze.count())

display(routes_bronze.orderBy(F.desc("shipment_count")))

Bronze Routing table created!
Total routes: 50


customer_zip,distance_meters,duration_seconds,osrm_code,seller_zip,shipment_count,distance_km,estimated_duration_hours
28970,851749.4,40250.4,Ok,14940,15,851.75,11.18
80030,14495.6,1243.8,Ok,81130,15,14.5,0.35
35162,867785.3,43627.1,Ok,14940,14,867.79,12.12
22793,752730.5,34977.5,Ok,14940,13,752.73,9.72
13212,288733.6,13667.8,Ok,14940,12,288.73,3.8
22775,740259.6,34532.7,Ok,14940,10,740.26,9.59
11065,418064.8,19989.5,Ok,14940,10,418.06,5.55
13290,275814.9,13303.4,Ok,14940,9,275.81,3.7
33200,676707.8,32088.4,Ok,14940,9,676.71,8.91
29101,1177466.0,62222.1,Ok,14940,9,1177.47,17.28
